# 19 · Optimiseurs — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/ch19_optimiseurs/03_notebook.ipynb)

La partie 0 vérifie tes réponses courtes aux quiz, aux rappels et aux exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wb.check`, ou les tests de ta librairie `mylearn`). « Exécuter tout » va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Les questions « dans tes notes » qui n'ont pas de cellule 📝 se notent dans la section « Notes sur le notebook » de ta copie de `06_mes_reponses.md`. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/ch19_optimiseurs/03_notebook.ipynb`, créée par `python tools/start_chapter.py 19`) : ce fichier-ci est mis à jour par Claude.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | 19.Q1 à 19.Q12, 19.R1 à 19.R3, 19.1 à 19.6, 19.9 | vérifier tes réponses courtes | 🧠 🔁 ✏️ 🧮 | ★ à ★★ | |
| A | 19.14 à 19.19 | un pas constant sur une vallée, puis ta librairie optim.py : plannings, SGD, Momentum et Nesterov, Adagrad, RMSprop et Adadelta, Adam et AdamW, warmup et cosinus, clipping (*prochaine session de génération*) | | | |
| B | 19.20 à 19.22 | trop de momentum, les figures du chapitre, un Adam qui diverge (*prochaine session de génération*) | | | |
| C | 19.23 à 19.25 | lire des courbes de loss d'optimiseurs, les solveurs de MLPClassifier, batch contre stochastique contre mini-batch sur les lunes (*prochaine session de génération*) | | | |
| D | 19.26 à 19.28 | une course d'optimiseurs sur la fonction de Rosenbrock, huit optimiseurs sur MNIST, le défi des 98 % en cinq epochs (*prochaine session de génération*) | | | |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="19")  # your own library

## Objectifs et rappel express

- Calculer à la main les premiers pas d'un planning, du momentum, de Nesterov, d'Adagrad, de RMSprop et d'Adam.
- Écrire et vérifier contre `torch.optim` les optimiseurs et les plannings de `mylearn.optim`.
- Comparer des optimiseurs sur des vallées, sur la fonction de Rosenbrock et sur MNIST, et défendre un choix.

**Rappel express** (conventions de PyTorch, $g$ le gradient, $t$ compté à partir de 1). Momentum : $b = \mu b + g$ ($b_1 = g_1$), $w \leftarrow w - \eta b$ ; Nesterov : $w \leftarrow w - \eta(g + \mu b)$ ; Adagrad : $s \leftarrow s + g^2$, $w \leftarrow w - \eta g/(\sqrt{s} + \epsilon)$ ; RMSprop : $v \leftarrow \alpha v + (1 - \alpha) g^2$, $w \leftarrow w - \eta g/(\sqrt{v} + \epsilon)$ ; Adam : $m \leftarrow \beta_1 m + (1 - \beta_1) g$, $v \leftarrow \beta_2 v + (1 - \beta_2) g^2$, $w \leftarrow w - \eta\,\frac{m/(1 - \beta_1^t)}{\sqrt{v/(1 - \beta_2^t)} + \epsilon}$ ; AdamW : $w \leftarrow w(1 - \eta\lambda)$ puis le pas d'Adam. Plannings : $\eta_0\gamma^k$, $\eta_0\gamma^{\lfloor k/s \rfloor}$, warmup $\eta_{\max} t/T_w$ puis cosinus.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ 19.1 à 19.6 et 🧮 19.9)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"E"`) ; une forme : entre guillemets (`"(3, 5)"`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », les preuves ∂ 19.7 et 19.8, la réflexion (🗣️ 19.10, ⚖️ 19.11, 📄 19.12, 🛠️ 19.13) et l'entretien se corrigent avec `05_solutions.md`.

**19.Q1 — Qu'appelle-t-on un optimiseur ?**

In [ ]:
# 19.Q1: the values you found on paper
answer_19_Q1a = ...  # a) True or False
answer_19_Q1b = ...  # b) the letter of your choice, e.g. "E"
answer_19_Q1c = ...  # c) True or False
answer_19_Q1d = ...  # d) the letter of your choice, e.g. "E"

for letter, answer in zip("abcd", [answer_19_Q1a, answer_19_Q1b, answer_19_Q1c, answer_19_Q1d]):
    wb.check(f"19.Q1{letter}", answer)

**19.Q2 — Minimum, maximum, plateau, selle : où le gradient s'annule-t-il ?**

In [ ]:
# 19.Q2: the values you found on paper
answer_19_Q2a = ...  # a) the letter of your choice, e.g. "E"
answer_19_Q2b = ...  # b) the letter of your choice, e.g. "E"
answer_19_Q2c = ...  # c) the letter of your choice, e.g. "E"
answer_19_Q2d = ...  # d) the letter of your choice, e.g. "E"
answer_19_Q2e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_19_Q2a, answer_19_Q2b, answer_19_Q2c, answer_19_Q2d, answer_19_Q2e]):
    wb.check(f"19.Q2{letter}", answer)

**19.Q3 — Minima locaux ou points selles : le vrai piège en grande dimension**

In [ ]:
# 19.Q3: the values you found on paper
answer_19_Q3a = ...  # a) the letter of your choice, e.g. "E"
answer_19_Q3b = ...  # b) a whole number: the probability is 1 over …
answer_19_Q3c = ...  # c) True or False
answer_19_Q3d = ...  # d) True or False

for letter, answer in zip("abcd", [answer_19_Q3a, answer_19_Q3b, answer_19_Q3c, answer_19_Q3d]):
    wb.check(f"19.Q3{letter}", answer)

**19.Q4 — Pourquoi lisser une erreur de comptage ?**

In [ ]:
# 19.Q4: the values you found on paper
answer_19_Q4a = ...  # a) True or False
answer_19_Q4b = ...  # b) True or False
answer_19_Q4c = ...  # c) the letter of your choice, e.g. "E"
answer_19_Q4d = ...  # d) the letter of your choice, e.g. "E"

for letter, answer in zip("abcd", [answer_19_Q4a, answer_19_Q4b, answer_19_Q4c, answer_19_Q4d]):
    wb.check(f"19.Q4{letter}", answer)

**19.Q5 — Pas constant : pourquoi rebondit-on au fond de la vallée ?**

In [ ]:
# 19.Q5: the values you found on paper
answer_19_Q5a = ...  # a) a number
answer_19_Q5b = ...  # b) True or False
answer_19_Q5c = ...  # c) the letter of your choice, e.g. "E"
answer_19_Q5d = ...  # d) the letter of your choice, e.g. "E"
answer_19_Q5e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_19_Q5a, answer_19_Q5b, answer_19_Q5c, answer_19_Q5d, answer_19_Q5e]):
    wb.check(f"19.Q5{letter}", answer)

**19.Q6 — Décroissance exponentielle : que vaut η après 50 pas ?**

In [ ]:
# 19.Q6: the values you found on paper
answer_19_Q6a = ...  # a) a number, 4 decimals
answer_19_Q6b = ...  # b) a whole number
answer_19_Q6c = ...  # c) the letter of your choice, e.g. "E"
answer_19_Q6d = ...  # d) True or False
answer_19_Q6e = ...  # e) a number

for letter, answer in zip("abcde", [answer_19_Q6a, answer_19_Q6b, answer_19_Q6c, answer_19_Q6d, answer_19_Q6e]):
    wb.check(f"19.Q6{letter}", answer)

**19.Q7 — Plannings : exponentiel, retardé, par paliers, sur l'erreur, bold driver**

In [ ]:
# 19.Q7: the values you found on paper
answer_19_Q7a = ...  # a) the letter of your choice, e.g. "E"
answer_19_Q7b = ...  # b) the letter of your choice, e.g. "E"
answer_19_Q7c = ...  # c) the letter of your choice, e.g. "E"
answer_19_Q7d = ...  # d) the letter of your choice, e.g. "E"
answer_19_Q7e = ...  # e) the letter of your choice, e.g. "E"
answer_19_Q7f = ...  # f) True or False

for letter, answer in zip("abcdef", [answer_19_Q7a, answer_19_Q7b, answer_19_Q7c, answer_19_Q7d, answer_19_Q7e, answer_19_Q7f]):
    wb.check(f"19.Q7{letter}", answer)

**19.Q8 — Batch, stochastique, mini-batch : compter les mises à jour**

In [ ]:
# 19.Q8: the values you found on paper
answer_19_Q8a = ...  # a) a whole number
answer_19_Q8b = ...  # b) a whole number
answer_19_Q8c = ...  # c) a whole number
answer_19_Q8d = ...  # d) a whole number
answer_19_Q8e = ...  # e) the letter of your choice, e.g. "E"
answer_19_Q8f = ...  # f) True or False

for letter, answer in zip("abcdef", [answer_19_Q8a, answer_19_Q8b, answer_19_Q8c, answer_19_Q8d, answer_19_Q8e, answer_19_Q8f]):
    wb.check(f"19.Q8{letter}", answer)

**19.Q9 — Momentum : que se passe-t-il si γ = 0 ? si γ = 1 ?**

In [ ]:
# 19.Q9: the values you found on paper
answer_19_Q9a = ...  # a) the letter of your choice, e.g. "E"
answer_19_Q9b = ...  # b) a number
answer_19_Q9c = ...  # c) the letter of your choice, e.g. "E"
answer_19_Q9d = ...  # d) the letter of your choice, e.g. "E"
answer_19_Q9e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_19_Q9a, answer_19_Q9b, answer_19_Q9c, answer_19_Q9d, answer_19_Q9e]):
    wb.check(f"19.Q9{letter}", answer)

**19.Q10 — Nesterov : regarder là où l'on va**

In [ ]:
# 19.Q10: the values you found on paper
answer_19_Q10a = ...  # a) the letter of your choice, e.g. "E"
answer_19_Q10b = ...  # b) True or False
answer_19_Q10c = ...  # c) True or False
answer_19_Q10d = ...  # d) the letter of your choice, e.g. "E"

for letter, answer in zip("abcd", [answer_19_Q10a, answer_19_Q10b, answer_19_Q10c, answer_19_Q10d]):
    wb.check(f"19.Q10{letter}", answer)

**19.Q11 — Adagrad, Adadelta, RMSprop : un learning rate par poids**

In [ ]:
# 19.Q11: the values you found on paper
answer_19_Q11a = ...  # a) True or False
answer_19_Q11b = ...  # b) the letter of your choice, e.g. "E"
answer_19_Q11c = ...  # c) a number
answer_19_Q11d = ...  # d) the letter of your choice, e.g. "E"
answer_19_Q11e = ...  # e) a number
answer_19_Q11f = ...  # f) True or False

for letter, answer in zip("abcdef", [answer_19_Q11a, answer_19_Q11b, answer_19_Q11c, answer_19_Q11d, answer_19_Q11e, answer_19_Q11f]):
    wb.check(f"19.Q11{letter}", answer)

**19.Q12 — Adam, No Free Lunch et choix d'un optimiseur**

In [ ]:
# 19.Q12: the values you found on paper
answer_19_Q12a = ...  # a) the letter of your choice, e.g. "E"
answer_19_Q12b = ...  # b) a number
answer_19_Q12c = ...  # c) a number
answer_19_Q12d = ...  # d) True or False
answer_19_Q12e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_19_Q12a, answer_19_Q12b, answer_19_Q12c, answer_19_Q12d, answer_19_Q12e]):
    wb.check(f"19.Q12{letter}", answer)

**19.R1 — Du delta au gradient d'un poids**

In [ ]:
# 19.R1: the values you found on paper
answer_19_R1a = ...  # a) a number
answer_19_R1b = ...  # b) a number
answer_19_R1c = ...  # c) a shape as a string, e.g. "(3, 5)"
answer_19_R1d = ...  # d) True or False

for letter, answer in zip("abcd", [answer_19_R1a, answer_19_R1b, answer_19_R1c, answer_19_R1d]):
    wb.check(f"19.R1{letter}", answer)

**19.R2 — Pourquoi l'initialisation He garde la variance stable**

In [ ]:
# 19.R2: the values you found on paper
answer_19_R2a = ...  # a) a number
answer_19_R2b = ...  # b) a number
answer_19_R2c = ...  # c) a number
answer_19_R2d = ...  # d) the letter of your choice, e.g. "E"

for letter, answer in zip("abcd", [answer_19_R2a, answer_19_R2b, answer_19_R2c, answer_19_R2d]):
    wb.check(f"19.R2{letter}", answer)

**19.R3 — Standardiser les features, et l'effet sur la descente**

In [ ]:
# 19.R3: the values you found on paper
answer_19_R3a = ...  # a) a whole number
answer_19_R3b = ...  # b) a number of millionths (a multiple of 1e-6)
answer_19_R3c = ...  # c) a whole number
answer_19_R3d = ...  # d) True or False

for letter, answer in zip("abcd", [answer_19_R3a, answer_19_R3b, answer_19_R3c, answer_19_R3d]):
    wb.check(f"19.R3{letter}", answer)

**Ex 19.1 — Plannings exponentiel et par paliers à la main**

In [ ]:
# Ex 19.1: the values you found on paper
answer_19_1a = ...  # a) a number, 4 decimals
answer_19_1b = ...  # b) a number, 4 decimals
answer_19_1c = ...  # c) a number
answer_19_1d = ...  # d) a number
answer_19_1e = ...  # e) a number
answer_19_1f = ...  # f) a number, 4 decimals
answer_19_1g = ...  # g) a number, 4 decimals

for letter, answer in zip("abcdefg", [answer_19_1a, answer_19_1b, answer_19_1c, answer_19_1d, answer_19_1e, answer_19_1f, answer_19_1g]):
    wb.check(f"19.1{letter}", answer)

**Ex 19.2 — Trois pas de momentum, deux conventions**

In [ ]:
# Ex 19.2: the values you found on paper
answer_19_2a = ...  # a) a number
answer_19_2b = ...  # b) a number
answer_19_2c = ...  # c) a number
answer_19_2d = ...  # d) a number
answer_19_2e = ...  # e) a number, 3 decimals
answer_19_2f = ...  # f) True or False
answer_19_2g = ...  # g) a number
answer_19_2h = ...  # h) a number

for letter, answer in zip("abcdefgh", [answer_19_2a, answer_19_2b, answer_19_2c, answer_19_2d, answer_19_2e, answer_19_2f, answer_19_2g, answer_19_2h]):
    wb.check(f"19.2{letter}", answer)

**Ex 19.3 — Nesterov : gradient anticipé et reformulation de PyTorch**

In [ ]:
# Ex 19.3: the values you found on paper
answer_19_3a = ...  # a) a number
answer_19_3b = ...  # b) a number
answer_19_3c = ...  # c) a number
answer_19_3d = ...  # d) a number, 4 decimals
answer_19_3e = ...  # e) a number
answer_19_3f = ...  # f) a number, 4 decimals
answer_19_3g = ...  # g) the letter of your choice, e.g. "E"

for letter, answer in zip("abcdefg", [answer_19_3a, answer_19_3b, answer_19_3c, answer_19_3d, answer_19_3e, answer_19_3f, answer_19_3g]):
    wb.check(f"19.3{letter}", answer)

**Ex 19.4 — Adagrad et RMSprop sur une suite de gradients**

In [ ]:
# Ex 19.4: the values you found on paper
answer_19_4a = ...  # a) a number
answer_19_4b = ...  # b) a number, 4 decimals
answer_19_4c = ...  # c) a number, 4 decimals
answer_19_4d = ...  # d) a number
answer_19_4e = ...  # e) a number, 4 decimals
answer_19_4f = ...  # f) a number
answer_19_4g = ...  # g) a number, 4 decimals
answer_19_4h = ...  # h) a number, 4 decimals
answer_19_4i = ...  # i) a number

for letter, answer in zip("abcdefghi", [answer_19_4a, answer_19_4b, answer_19_4c, answer_19_4d, answer_19_4e, answer_19_4f, answer_19_4g, answer_19_4h, answer_19_4i]):
    wb.check(f"19.4{letter}", answer)

**Ex 19.5 — Un pas d'Adam et la correction de biais**

In [ ]:
# Ex 19.5: the values you found on paper
answer_19_5a = ...  # a) a number
answer_19_5b = ...  # b) a number, 5 decimals
answer_19_5c = ...  # c) a number
answer_19_5d = ...  # d) a number
answer_19_5e = ...  # e) a number, 4 decimals
answer_19_5f = ...  # f) a number
answer_19_5g = ...  # g) a number, 4 decimals
answer_19_5h = ...  # h) a number, 5 decimals
answer_19_5i = ...  # i) True or False
answer_19_5j = ...  # j) True or False

for letter, answer in zip("abcdefghij", [answer_19_5a, answer_19_5b, answer_19_5c, answer_19_5d, answer_19_5e, answer_19_5f, answer_19_5g, answer_19_5h, answer_19_5i, answer_19_5j]):
    wb.check(f"19.5{letter}", answer)

**Ex 19.6 — Adam + L2 ou AdamW : la même pénalité ? Convertir alpha, lambda et weight_decay**

In [ ]:
# Ex 19.6: the values you found on paper
answer_19_6a = ...  # a) a number
answer_19_6b = ...  # b) a number
answer_19_6c = ...  # c) a number
answer_19_6d = ...  # d) a number, 5 decimals
answer_19_6e = ...  # e) a number, 4 decimals
answer_19_6f = ...  # f) a number
answer_19_6g = ...  # g) a number, 5 decimals
answer_19_6h = ...  # h) a number
answer_19_6i = ...  # i) a number of 1e-7
answer_19_6j = ...  # j) True or False

for letter, answer in zip("abcdefghij", [answer_19_6a, answer_19_6b, answer_19_6c, answer_19_6d, answer_19_6e, answer_19_6f, answer_19_6g, answer_19_6h, answer_19_6i, answer_19_6j]):
    wb.check(f"19.6{letter}", answer)

**Ex 19.9 — Fermi : la mémoire des optimiseurs, du MLP au LLM**

In [ ]:
# Ex 19.9: the values you found on paper
answer_19_9a = ...  # a) a whole number
answer_19_9b = ...  # b) a number of MB, 1 decimal
answer_19_9c = ...  # c) a number of MB, 1 decimal
answer_19_9d = ...  # d) a number of MB, 1 decimal
answer_19_9e = ...  # e) a whole number of GB
answer_19_9f = ...  # f) a whole number of GB
answer_19_9g = ...  # g) a whole number

for letter, answer in zip("abcdefg", [answer_19_9a, answer_19_9b, answer_19_9c, answer_19_9d, answer_19_9e, answer_19_9f, answer_19_9g]):
    wb.check(f"19.9{letter}", answer)

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu dérouler à la main deux pas de momentum, de RMSprop et d'Adam, correction de biais comprise ?
2. Sais-tu expliquer pourquoi la courbure limite le learning rate, et ce que change une vallée étroite ?
3. Sais-tu dire la différence entre Adam avec `weight_decay` et AdamW, et ce que font le warmup, le cosinus et le clipping ?

**Pour aller plus loin** : la synthèse de S. Ruder et l'article interactif de G. Goh sur le momentum, cités dans la fiche. La suite : le ch. 20, où PyTorch fera tout cela avec `torch.optim`.

*Les exercices de code 19.14 à 19.28 (parties A à D, ta librairie `optim.py`) seront ajoutés à la prochaine session de génération ; relance alors `python tools/start_chapter.py 19` pour obtenir le notebook complet.*